# Wine Quality — Exploratory Data Analysis (EDA)

### Machine Learning Classification Project

This notebook performs Exploratory Data Analysis on the **UCI Wine Quality red-wine dataset** used in the project.

**Dataset:** 1,599 red-wine samples  
**Features:** 11 physicochemical properties  
**Target:** Wine quality score (3–8)

The analysis focuses on understanding the dataset, checking data quality, studying feature distributions, identifying outliers, examining class imbalance, and exploring relationships between physicochemical properties and wine quality.


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

print("Libraries imported successfully.")


## 2. Load the Dataset

Place the red-wine CSV file in the same folder as this notebook and name it:

`winequality-red.csv`

The notebook also searches the current directory for a CSV containing a `quality` column if the expected filename is not found.


In [ ]:
DATA_FILE = Path("winequality-red.csv")

if DATA_FILE.exists():
    df = pd.read_csv(DATA_FILE, sep=";")
else:
    candidates = list(Path(".").glob("*.csv"))
    df = None
    for file in candidates:
        try:
            temp = pd.read_csv(file, sep=";")
            if "quality" in temp.columns:
                df = temp
                DATA_FILE = file
                break
        except Exception:
            pass

    if df is None:
        raise FileNotFoundError(
            "winequality-red.csv was not found. Place the UCI Wine Quality red-wine CSV "
            "in the notebook folder and run this cell again."
        )

print(f"Loaded: {DATA_FILE}")
print(f"Dataset shape: {df.shape}")
df.head()


## 3. Dataset Overview

In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()


In [ ]:
df.head(10)


### Feature Description

The dataset contains 11 measurable physicochemical properties and one target variable.

- **Fixed acidity**
- **Volatile acidity**
- **Citric acid**
- **Residual sugar**
- **Chlorides**
- **Free sulfur dioxide**
- **Total sulfur dioxide**
- **Density**
- **pH**
- **Sulphates**
- **Alcohol**
- **Quality** — sensory quality score


## 4. Data Quality Check

In [ ]:
print("Missing values:")
print(df.isnull().sum())

print("\nTotal missing values:", df.isnull().sum().sum())

print("\nDuplicate rows:", df.duplicated().sum())


In [ ]:
df.describe().T


### Interpretation

The project PPT reports **0 missing values** in the dataset. The quality scores present in the project are concentrated between **3 and 8**.

The descriptive statistics below help identify the central tendency, spread, minimum, and maximum values of every numeric variable.


## 5. Wine Quality Distribution

In [ ]:
quality_counts = df["quality"].value_counts().sort_index()

plt.figure(figsize=(9, 5))
ax = sns.barplot(x=quality_counts.index, y=quality_counts.values)
plt.title("Wine Quality Distribution")
plt.xlabel("Quality Score")
plt.ylabel("Number of Samples")

for i, value in enumerate(quality_counts.values):
    ax.text(i, value + max(quality_counts.values) * 0.01, str(value),
            ha="center", va="bottom")

plt.tight_layout()
plt.show()

print(quality_counts)


### Key Finding

The quality distribution is strongly imbalanced. In the project dataset, quality **5 and 6 contain the large majority of observations**, while the extreme scores such as **3, 4, and 8 are rare**.

This imbalance is important for classification because a model can perform well on the common classes while struggling with the rare quality classes.


## 6. Distribution of Physicochemical Features

In [ ]:
feature_cols = [c for c in df.columns if c != "quality"]

df[feature_cols].hist(figsize=(16, 12), bins=25, edgecolor="black")
plt.suptitle("Distribution of Physicochemical Features", fontsize=16)
plt.tight_layout()
plt.show()


## 7. Boxplots for Outlier Analysis

In [ ]:
plt.figure(figsize=(16, 10))
df[feature_cols].boxplot(rot=45)
plt.title("Boxplots of Physicochemical Features")
plt.ylabel("Value")
plt.tight_layout()
plt.show()


### Outlier Analysis

Several physicochemical variables contain observations far from their central ranges. The project preprocessing approach uses the **IQR method to flag extreme values and cap them rather than dropping observations**, preserving the dataset size.


## 8. Volatile Acidity vs. Wine Quality

In [ ]:
mean_volatile = df.groupby("quality")["volatile acidity"].mean()

plt.figure(figsize=(9, 5))
ax = sns.barplot(x=mean_volatile.index, y=mean_volatile.values)
plt.title("Mean Volatile Acidity by Wine Quality")
plt.xlabel("Quality Score")
plt.ylabel("Mean Volatile Acidity (g/dm³)")

for i, value in enumerate(mean_volatile.values):
    ax.text(i, value + 0.01, f"{value:.2f}", ha="center")

plt.tight_layout()
plt.show()

mean_volatile


### Key Finding

The project PPT identifies a **clear downward trend** in average volatile acidity as quality increases. Higher volatile acidity is associated with lower sensory quality scores.

This makes volatile acidity one of the strongest and most interpretable predictors in the project.


## 9. Alcohol vs. Wine Quality

In [ ]:
mean_alcohol = df.groupby("quality")["alcohol"].mean()

plt.figure(figsize=(9, 5))
ax = sns.barplot(x=mean_alcohol.index, y=mean_alcohol.values)
plt.title("Mean Alcohol Content by Wine Quality")
plt.xlabel("Quality Score")
plt.ylabel("Mean Alcohol (% vol.)")

for i, value in enumerate(mean_alcohol.values):
    ax.text(i, value + 0.05, f"{value:.2f}", ha="center")

plt.tight_layout()
plt.show()

mean_alcohol


### Key Finding

Alcohol has the **strongest positive Pearson correlation with quality (+0.48)** among the features reported in the project PPT.


## 10. Correlation Analysis

In [ ]:
corr = df.corr(numeric_only=True)

plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()


### Correlation of Features with Quality

In [ ]:
quality_corr = (
    df.corr(numeric_only=True)["quality"]
    .drop("quality")
    .sort_values(ascending=False)
)

plt.figure(figsize=(10, 6))
ax = sns.barplot(x=quality_corr.values, y=quality_corr.index)
plt.title("Pearson Correlation of Features with Wine Quality")
plt.xlabel("Correlation with Quality")
plt.ylabel("Feature")
plt.axvline(0, linewidth=1)

for i, value in enumerate(quality_corr.values):
    ax.text(value + (0.015 if value >= 0 else -0.015), i,
            f"{value:.2f}",
            va="center",
            ha="left" if value >= 0 else "right")

plt.tight_layout()
plt.show()

quality_corr


### Correlation Findings

According to the project analysis:

- **Alcohol (+0.48)** has the strongest positive relationship with quality.
- **Volatile acidity (−0.39)** has the strongest negative relationship.
- **Sulphates (+0.25)** and **citric acid (+0.23)** show smaller positive relationships.
- **Free sulfur dioxide and pH** have weak linear relationships with quality.
- Correlation measures linear association only; a low correlation does not necessarily mean a feature has no predictive value.


## 11. Quality by Alcohol and Volatile Acidity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.boxplot(data=df, x="quality", y="alcohol", ax=axes[0])
axes[0].set_title("Alcohol by Quality")
axes[0].set_xlabel("Quality Score")
axes[0].set_ylabel("Alcohol (% vol.)")

sns.boxplot(data=df, x="quality", y="volatile acidity", ax=axes[1])
axes[1].set_title("Volatile Acidity by Quality")
axes[1].set_xlabel("Quality Score")
axes[1].set_ylabel("Volatile Acidity (g/dm³)")

plt.tight_layout()
plt.show()


## 12. IQR-Based Outlier Capping

In [ ]:
def cap_outliers_iqr(data, columns):
    capped = data.copy()
    bounds = {}

    for col in columns:
        q1 = capped[col].quantile(0.25)
        q3 = capped[col].quantile(0.75)
        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        bounds[col] = (lower, upper)
        capped[col] = capped[col].clip(lower=lower, upper=upper)

    return capped, bounds

df_capped, iqr_bounds = cap_outliers_iqr(df, feature_cols)

print("Outlier capping completed using the IQR rule.")
df_capped.head()


## 13. Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(df_capped[feature_cols])
X_scaled = pd.DataFrame(X_scaled, columns=feature_cols)

print("Scaled feature means:")
print(X_scaled.mean().round(3))

print("\nScaled feature standard deviations:")
print(X_scaled.std().round(3))


### Preprocessing Used in the Project

The project preprocessing pipeline:

1. Checks data integrity and missing values.
2. Flags extreme values using the IQR method.
3. Caps extreme observations rather than dropping rows.
4. Standardizes all 11 numeric input features.
5. Uses the quality variable as the classification target.
6. Uses an 80/20 train-test split with stratification for the modeling stage.


## 14. EDA Summary

### Main Findings

| Area | Observation |
|---|---|
| Dataset size | 1,599 samples and 12 columns |
| Missing values | 0 reported in the project |
| Target | Wine quality score from 3 to 8 |
| Class distribution | Strongly concentrated at quality 5 and 6 |
| Positive relationship | Alcohol (+0.48) |
| Negative relationship | Volatile acidity (−0.39) |
| Other positive signals | Sulphates (+0.25), citric acid (+0.23) |
| Preprocessing | IQR capping and standardization |
| Modeling implication | Class imbalance makes multi-class prediction more difficult |

### Overall EDA Conclusion

The exploratory analysis shows that wine quality is not evenly distributed across the available quality scores. Most observations belong to the middle quality classes, while extreme scores are uncommon.

Among the physicochemical measurements, **alcohol** has the strongest positive linear relationship with quality, while **volatile acidity** has the strongest negative relationship reported in the project. These findings provide useful context for the classification models used later in the project.


## 15. Connection to the Classification Models

The EDA supports the modeling choices in the Wine Quality Classification project.

- **Binary Logistic Regression** groups the target into good vs. not-good wine.
- **Multinomial Logistic Regression** keeps multiple quality classes.
- The strong class imbalance observed during EDA helps explain why the multi-class problem is more challenging.
- Standardization ensures the 11 numeric features are placed on comparable scales before regression-based modeling.

The reported project results are:

| Metric | Logistic Regression | Multinomial Logistic Regression |
|---|---:|---:|
| Accuracy | 74% | 61% |
| Precision | 72% | 58% |
| Recall | 70% | 57% |
| F1-score | 71% | 57% |
| Classes | 2 | 6 |


# Final Conclusion

The Wine Quality EDA reveals a dataset with clean records but a strongly imbalanced target distribution. Quality scores 5 and 6 dominate the dataset, while scores 3, 4, and 8 occur much less frequently.

The analysis also identifies alcohol and volatile acidity as the most prominent linear relationships with quality in the project. Outlier treatment and feature standardization prepare the data for the classification stage.

These EDA findings provide the foundation for comparing binary Logistic Regression with Multinomial Logistic Regression for wine quality classification.
